# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Microsoft Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.


## Thiết lập

Trước khi chạy sổ tay này, hãy đảm bảo bạn đã:

1. **Một dự án Microsoft Foundry** với một mô hình chat đã được triển khai (ví dụ: `gpt-5-mini`).
2. **Đăng nhập bằng Azure CLI** — chạy `az login` trong terminal của bạn.
3. **Đặt các biến môi trường cần thiết:**
   - `AZURE_AI_PROJECT_ENDPOINT` — điểm cuối dự án Microsoft Foundry của bạn.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — tên của mô hình bạn đã triển khai.

Ô dưới đây cài đặt các gói Python bạn cần.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Tạo Đại lý Đầu tiên của Bạn

Một đại lý cần hai thứ:

- **Hướng dẫn** cho nó biết *nó là ai* và *cách cư xử* (một lệnh hệ thống).
- **Công cụ** — Các hàm Python được trang trí bằng `@tool` mà đại lý có thể gọi để lấy thông tin hoặc thực hiện hành động.

Dưới đây, chúng tôi định nghĩa một công cụ đơn giản trả về danh sách các điểm đến du lịch phổ biến. Đại lý sẽ sử dụng công cụ này khi người dùng yêu cầu đề xuất du lịch.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Phản hồi dạng phát trực tiếp

Để có trải nghiệm tương tác hơn, bạn có thể **phát trực tiếp** phản hồi của tác nhân. Thay vì chờ đợi câu trả lời hoàn chỉnh, tác nhân sẽ xuất các đoạn văn bản ngay khi chúng được tạo ra. Điều này đặc biệt hữu ích trong các giao diện trò chuyện, nơi bạn muốn hiển thị kết quả theo thời gian thực.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Tóm tắt

Trong bài học này, bạn đã học cách:

- **Tạo một nhà cung cấp** kết nối đến Microsoft Foundry Agent Service thông qua `FoundryChatClient`.
- **Định nghĩa một công cụ** sử dụng bộ trang trí `@tool` để agent có thể gọi các hàm Python của bạn.
- **Chạy agent** với một tin nhắn người dùng và in phản hồi của nó.
- **Phát trực tiếp phản hồi** để xuất ra thời gian thực.

Trong bài học tiếp theo, chúng ta sẽ khám phá sâu hơn về các framework agentic và học cách cung cấp cho agent các công cụ mạnh mẽ hơn cùng khả năng suy luận đa bước.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
